In [1]:
from pathlib import Path
import os
import ctypes
import pandas as pd
from functools import lru_cache

# Windows hidden attribute
FILE_ATTRIBUTE_HIDDEN = 0x2
FILE_ATTRIBUTE_SYSTEM = 0x4


def is_hidden_or_system(path: Path) -> bool:
    """
    判断 Windows 隐藏 / 系统文件夹。
    普通以 . 开头的隐藏命名也会识别。
    """
    try:
        if path.name.startswith("."):
            return True

        attrs = ctypes.windll.kernel32.GetFileAttributesW(str(path))
        if attrs == -1:
            return False

        return bool(attrs & (FILE_ATTRIBUTE_HIDDEN | FILE_ATTRIBUTE_SYSTEM))
    except Exception:
        return False


def format_size(size_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB", "PB"]
    size = float(size_bytes)

    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024

    return f"{size:.2f} EB"


@lru_cache(maxsize=None)
def folder_size(path_str: str) -> int:
    """
    递归计算文件夹总大小。
    包含隐藏文件、隐藏文件夹。
    遇到无权限文件会自动跳过。
    不跟随符号链接，避免循环扫描。
    """
    path = Path(path_str)
    total = 0

    try:
        with os.scandir(path) as entries:
            for entry in entries:
                try:
                    if entry.is_symlink():
                        continue

                    if entry.is_file(follow_symlinks=False):
                        total += entry.stat(follow_symlinks=False).st_size

                    elif entry.is_dir(follow_symlinks=False):
                        total += folder_size(entry.path)

                except (PermissionError, FileNotFoundError, OSError):
                    continue

    except (PermissionError, FileNotFoundError, OSError):
        pass

    return total


def top_child_folders(path: Path, top_n: int = 10):
    """
    获取某个路径下，按占用空间排序的前 top_n 个直接子文件夹。
    """
    folders = []

    try:
        with os.scandir(path) as entries:
            for entry in entries:
                try:
                    if entry.is_symlink():
                        continue

                    if entry.is_dir(follow_symlinks=False):
                        child_path = Path(entry.path)
                        size = folder_size(str(child_path))
                        folders.append({
                            "path": str(child_path),
                            "name": child_path.name,
                            "size_bytes": size,
                            "size": format_size(size),
                            "hidden_or_system": is_hidden_or_system(child_path),
                        })

                except (PermissionError, FileNotFoundError, OSError):
                    continue

    except (PermissionError, FileNotFoundError, OSError):
        pass

    folders.sort(key=lambda x: x["size_bytes"], reverse=True)
    return folders[:top_n]


def scan_top_folders(root_path: str, top_n: int = 10, depth: int = 3):
    """
    从 root_path 开始扫描：
    第 1 层：root_path 下前 top_n 大文件夹
    第 2 层：每个第 1 层文件夹内部前 top_n 大子文件夹
    第 3 层：每个第 2 层文件夹内部前 top_n 大子文件夹
    """
    root = Path(root_path)

    if not root.exists():
        raise FileNotFoundError(f"路径不存在：{root_path}")

    results = []

    def walk(current_path: Path, current_depth: int, parent: str | None):
        if current_depth > depth:
            return

        children = top_child_folders(current_path, top_n=top_n)

        for rank, item in enumerate(children, start=1):
            item["level"] = current_depth
            item["rank_in_parent"] = rank
            item["parent"] = parent
            results.append(item)

            walk(Path(item["path"]), current_depth + 1, item["path"])

    walk(root, current_depth=1, parent=str(root))

    df = pd.DataFrame(results)

    if not df.empty:
        df = df[
            [
                "level",
                "rank_in_parent",
                "name",
                "size",
                "size_bytes",
                "hidden_or_system",
                "parent",
                "path",
            ]
        ]

    return df


def print_folder_tree(df: pd.DataFrame):
    """
    用树状结构打印结果。
    """
    for _, row in df.iterrows():
        indent = "    " * (int(row["level"]) - 1)
        hidden_mark = " [hidden/system]" if row["hidden_or_system"] else ""
        print(
            f"{indent}{row['rank_in_parent']:>2}. "
            f"{row['name']} - {row['size']}{hidden_mark}"
        )

C:\Users\imgw\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
root_path = r"C:\Users\imgw"

df = scan_top_folders(
    root_path=root_path,
    top_n=10,
    depth=3
)

print_folder_tree(df)

df

 1. Documents - 370.14 GB
     1. Codex - 190.62 GB
         1. RTSPHEM-main - 86.38 GB
         2. geochemfoam - 38.96 GB
         3. SIP模拟 - 32.10 GB
         4. NMR-agent - 11.58 GB
         5. NMR_diffusion - 4.74 GB
         6. NMR模拟 - 3.67 GB
         7. T2agent - 3.50 GB
         8. 震电模拟 - 2.54 GB
         9. happy - 2.44 GB
        10. 纵向页岩 - 1.95 GB
     2. MATLAB - 132.13 GB
         1. RTSPHEM-main - 84.14 GB
         2. Batch_simu - 47.98 GB
         3. COMSOL_MATLAB_Interface - 176.86 KB
     3. Python - 18.20 GB
         1. 数据分析 - 16.13 GB
         2. ComfyUI_examples-master - 1.78 GB
         3. CrunchTope-master - 303.52 MB
     4. .tmp.driveupload - 9.87 GB [hidden/system]
     5. ComfyUI - 9.57 GB
         1. .venv - 9.25 GB [hidden/system]
         2. custom_nodes - 270.66 MB
         3. output - 42.10 MB
         4. user - 11.44 MB
         5. input - 5.33 MB
         6. models - 0.00 B
         7. temp - 0.00 B
     6. xwechat_files - 6.97 GB
         1. wxid_q4pzj

,level,rank_in_parent,name,size,size_bytes,hidden_or_system,parent,path
0,1,1,Documents,370.14 GB,397433241623,False,C:\Users\imgw,C:\Users\imgw\Documents
1,2,1,Codex,190.62 GB,204678428204,False,C:\Users\imgw\Documents,C:\Users\imgw\Documents\Codex
2,3,1,RTSPHEM-main,86.38 GB,92755112191,False,C:\Users\imgw\Documents\Codex,C:\Users\imgw\Documents\Codex\RTSPHEM-main
3,3,2,geochemfoam,38.96 GB,41837921606,False,C:\Users\imgw\Documents\Codex,C:\Users\imgw\Documents\Codex\geochemfoam
4,3,3,SIP模拟,32.10 GB,34462367617,False,C:\Users\imgw\Documents\Codex,C:\Users\imgw\Documents\Codex\SIP模拟
...,...,...,...,...,...,...,...,...
245,3,6,file_thumbnail,0.00 B,0,False,C:\Users\imgw\WPS Cloud Files\.309748709,C:\Users\imgw\WPS Cloud Files\.309748709\file_...
246,3,7,kdocsfile,0.00 B,0,False,C:\Users\imgw\WPS Cloud Files\.309748709,C:\Users\imgw\WPS Cloud Files\.309748709\kdocs...
247,3,8,metadata,0.00 B,0,False,C:\Users\imgw\WPS Cloud Files\.309748709,C:\Users\imgw\WPS Cloud Files\.309748709\metadata
248,3,9,syncfolderbackup,0.00 B,0,False,C:\Users\imgw\WPS Cloud Files\.309748709,C:\Users\imgw\WPS Cloud Files\.309748709\syncf...


In [4]:
root_path = r"C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\ReactiveTransport\Analysis"

df = scan_top_folders(
    root_path=root_path,
    top_n=10,
    depth=3
)

print_folder_tree(df)

df



 1. 孔隙耦合参数 - 10.82 GB
     1. original_data - 5.77 GB
         1. T2lm-single - 4.42 GB
         2. random-F-W - 1.35 GB
     2. original_data - Copy - 4.71 GB
         1. T2lm-single - 3.36 GB
         2. random-F-W - 1.35 GB
     3. overleafcn - 57.78 MB
     4. manuscript - 55.39 MB
         1. method - 55.13 MB
         2. figures - 265.63 KB
         3. 参考文献 - 0.00 B
     5. overleaf - 53.96 MB
     6. .venv-1 - 51.51 MB [hidden/system]
         1. Lib - 50.45 MB
         2. Scripts - 1.04 MB
         3. share - 14.80 KB
     7. Luisa_Internship_Project__Copy_ - 27.60 MB
         1. Data - 26.18 MB
     8. paper - 23.61 MB
         1. 故事线梳理 - 18.00 KB
     9. images - 23.15 MB
         1. fig1_combined_hex_random_parula - 5.54 MB
         2. fig1_random_parula - 2.64 MB
         3. fig1_hex_parula - 1.17 MB
    10. 师兄文章 - 15.04 MB
 2. hex_outputs - 3.01 MB
 3. random_outputs - 2.92 MB
 4. outputs - 2.70 MB
     1. T2pf_random - 2.70 MB
 5. Perm_data - 511.44 KB
     1. T2pf-random

,level,rank_in_parent,name,size,size_bytes,hidden_or_system,parent,path
0,1,1,孔隙耦合参数,10.82 GB,11615569919,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
1,2,1,original_data,5.77 GB,6194337940,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
2,3,1,T2lm-single,4.42 GB,4744723674,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
3,3,2,random-F-W,1.35 GB,1449614266,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
4,2,2,original_data - Copy,4.71 GB,5061029953,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
5,3,1,T2lm-single,3.36 GB,3611415687,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
6,3,2,random-F-W,1.35 GB,1449614266,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
7,2,3,overleafcn,57.78 MB,60581858,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
8,2,4,manuscript,55.39 MB,58082354,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
9,3,1,method,55.13 MB,57810344,False,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...,C:\Users\imgw\Documents\MATLAB\RTSPHEM-main\Re...
